In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('UNSTRUCTURED_API_KEY')
if api_key:
	os.environ["UNSTRUCTURED_API_KEY"] = api_key
else:
	print("Warning: UNSTRUCTURED_API_KEY environment variable not set")

In [2]:
import pickle
from pathlib import Path

cache_path = Path("../data/processed/docs_cache.pkl")
evaluation_file_path = "../data/raw/DBMS Chapter 7.pdf"

# Load from cache if exists
if cache_path.exists():
    with open(cache_path, 'rb') as f:
        docs = pickle.load(f)
    print("Loaded from cache")
else:
    # Load via API
    from langchain_unstructured import UnstructuredLoader
    loader = UnstructuredLoader(
        file_path=evaluation_file_path,
        api_key=api_key,
        partition_via_api=True,
    )
    docs = loader.load()
    
    # Save to cache
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, 'wb') as f:
        pickle.dump(docs, f)
    print("Loaded from API and cached")

Loaded from cache


In [3]:
import pickle
from pathlib import Path

# Save loaded documents
cache_path = Path("../data/processed/docs_cache.pkl")
cache_path.parent.mkdir(parents=True, exist_ok=True)

with open(cache_path, 'wb') as f:
    pickle.dump(docs, f)

print(f"Documents saved to {cache_path}")

Documents saved to ../data/processed/docs_cache.pkl


In [4]:
len(docs), docs[1].page_content

(355,
 'By: Er. Raj Kiran Chhatkuli Assistant Professor Department of Electronics & Computer Engineering IOE Paschimanchal Campus, TU')

In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv('GROQ_API_KEY'),
    temperature=0
)

# response = llm.invoke("Explain RAG in one sentence.")
# print(response.content)


/home/bigyan/Desktop/mero_pdf/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Encoding a query
query = "What is the difference between 2NF and 3NF?"
query_embedding = embedding_model.embed_query(query)

# Encoding chunks
chunks = [
    "2NF eliminates partial dependencies...",
    "3NF additionally removes transitive dependencies...",
    "An ER diagram represents entities...",
]
chunk_embeddings = embedding_model.embed_documents(chunks)

/tmp/ipykernel_112563/3103936423.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6435.36it/s]


In [7]:
len(chunk_embeddings), len(chunk_embeddings[0]) if chunk_embeddings else 0

(3, 768)

In [8]:
# Merge tiny documents and split into chunks-
from langchain_core.documents import Document
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200

# Merge all non-empty page content
full_text = "\n".join([doc.page_content.strip() for doc in docs if doc.page_content.strip() != ""])
docs_to_split = Document(page_content=full_text)
print(f"Merged {len(docs)} docs into 1 document with {len(full_text)} characters")

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", "? ", "! ", "; ", " ", ""],
    length_function=len,
)

# Split the merged document
chunked_docs = text_splitter.split_documents([docs_to_split])
print(f"Split merged document into {len(chunked_docs)} chunks")

# Notes:
# - Merging first avoids creating too many tiny chunks for tiny documents
# - Larger chunks work better for similarity search (we usually select only a few chunks)
# - Splitting after merging creates manageable, meaningful pieces


Merged 355 docs into 1 document with 30169 characters
Split merged document into 30 chunks


In [9]:
system_prompt = """"
You are a precise academic assistant that answers questions strictly based on the provided study material.

Your behavior rules:
- Answer ONLY using information from the context provided. Do not use any external knowledge.
- If the answer is not found in the context, respond exactly with: "This topic is not covered in the provided material."
- Never guess, infer beyond what is stated, or fill gaps with general knowledge.
- Use the same terminology and phrasing that appears in the source material.
- Keep answers concise and academically accurate — 2 to 4 sentences unless the question requires more detail.
- If a question is vague or ambiguous, ask one clarifying question before answering.
- Do not repeat the question back to the user.
- When comparing concepts, structure your answer clearly with each concept addressed separately.
"""

In [10]:
from langchain_community.vectorstores import Qdrant

# Create vectorstore and add documents
vectorstore = Qdrant.from_documents(
    documents=chunked_docs,
    embedding=embedding_model,
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    collection_name="pdf_chunks",
    force_recreate=True,  # Recreate if exists
)

print(f"Added {len(chunked_docs)} documents to Qdrant!")

Added 30 documents to Qdrant!


In [11]:
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":2}
)

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "Context:{context} Input:{input}")
])

In [13]:
from langchain_classic.chains import create_history_aware_retriever

retriever_prompt = ChatPromptTemplate.from_messages([
    ("system","Rewrite the user query using the chat history if needed. Do NOT answer."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

In [14]:
history_aware_retriever = create_history_aware_retriever(
    llm,retriever,retriever_prompt
)

In [15]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
question_answer_chain = create_stuff_documents_chain(
    llm,prompt=answer_prompt
)

In [16]:
from langchain_core.runnables import RunnableParallel
rag_chain = (
    RunnableParallel({
        'input':lambda d: d['input'],
        'chat_history':lambda d: d['chat_history'],
        'context': lambda d:history_aware_retriever.invoke({
            'chat_history': d['chat_history'],
            'input': d['input'],
        })
    })
    | question_answer_chain
)

## Evaluation

In [17]:
# preparation of dataset for the evaluation

questions = [
    #factual questions (10)
    "Define what a \"transaction\" is in a database system.",
    "What are the four ACID properties in DBMS?",
    "Explain the \"All or nothing rule\" as it relates to database transactions.",
    "What are the five states of a transaction in the simple transaction model?",
    "Define the \"Active\" state of a transaction.",
    "What is a \"Schedule\" in the context of DBMS?",
    "Under what conditions is a pair of operations considered to be \"conflicting\"?",
    "List the three main types of concurrency control protocols mentioned in the document.",
    "What is the difference between a Shared Lock (S) and an Exclusive Lock (X)?",
    "What is the fundamental rule of the Two-Phase Locking (2PL) Protocol?",
    #conceptual questions (5)
    "Why is the property of \"Isolation\" important for concurrent transactions?",
    "Compare and contrast \"Serial\" and \"Serializable\" schedules.",
    "Why might a database administrator choose to use \"View Serializability\" over \"Conflict Serializability\"?",
    "What are the primary trade-offs or disadvantages of strictly implementing ACID properties?",
    "Explain how \"Strict Two-Phase Locking\" (Strict-2PL) differs from standard \"Two-Phase Locking\" (2PL).",
    #application questions (5)
    "A system failure occurs after a bank transfer deducts money from Account A but before it adds it to Account B. Which ACID property handles this, and what is the outcome?",
    "Two airline agents see one remaining seat and both try to book it at the exact same time. How does 2PL prevent them from both successfully selling the same seat?",
    "A manager runs a report to sum all account balances while a transfer is moving $20 from Account A to Account B. If the manager sees the old balance for A but the new balance for B, what concurrency problem has occurred?",
    "Using the \"Shopping Trip\" analogy provided in the text, explain the transition from the \"Growing Phase\" to the \"Shrinking Phase\" in a transaction.",
    "If Transaction 1 holds a lock on Data A and waits for Data B, while Transaction 2 holds a lock on Data B and waits for Data A, what state is the system in and how can it be resolved?",
    #adversarial questions (5)
    "What are the specific rules for converting a database schema from Second Normal Form (2NF) to Third Normal Form (3NF)?",
    "How do you write a SQL query using the \"GROUP BY\" and \"HAVING\" clauses to filter aggregate results?",
    "Explain the difference between B-Tree and Hash indexing and when to use each for optimizing queries.",
    "What is the difference between a \"Natural Join\" and an \"Outer Join\" in relational algebra?",
    "Describe the Grant and Revoke commands used for database security and authorization.",
]

expected_answers = [
    "A transaction is a single logical unit of work that accesses and possibly modifies the contents of a database using read and write operations. It must maintain ACID properties to ensure accuracy, completeness, and data integrity.",
    "The ACID properties are Atomicity, Consistency, Isolation, and Durability. These properties ensure that the database remains accurate and integrated before and after a transaction.",
    "This rule refers to the Atomicity property, which states that a transaction must be treated as an atomic unit. This means either the entire transaction takes place at once or it does not happen at all, with no partial execution allowed.",
    "The five states are Active, Partially Committed, Failed, Aborted, and Committed. A transaction moves through these states from its initial execution until it is either permanently saved or rolled back.",
    "The Active state is the initial state of every transaction where the actual operations, such as updating, inserting, or deleting records, are being executed. At this stage, changes are being processed but are not yet saved to the database.",
    "A schedule is a series of operations from one transaction to another that is used to preserve the order of operations within individual transactions. Schedules can be categorized as serial or non-serial.",
    "A pair of operations is said to conflict if they operate on the same data item and at least one of them is a write operation. For example, a Read(X) and a Write(X) from different transactions would be conflicting.",
    "The three types of protocols are Lock-Based Protocols, Time-Based Protocols, and Validation-Based Protocols. These rules are implemented to maintain consistency and serializability during concurrent executions.",
    "A Shared Lock (S), or Read Lock, allows multiple transactions to read a data item simultaneously but disables write operations. An Exclusive Lock (X), or Write Lock, allows a single transaction to both read and write an item while preventing any other locks from being placed on it.",
    "The fundamental rule of 2PL is that a transaction must acquire all the locks it needs before it starts releasing any of them. This requirement ensures the serializability of the transaction schedule.",
    "Isolation ensures that multiple transactions occurring concurrently do not interfere with one another, preventing database inconsistency. It makes it so that changes in one transaction are not visible to others until they are officially committed, mimicking serial execution.",
    "A serial schedule executes transactions one after another, which is always correct but slow and inefficient. A serializable schedule allow transactions to run concurrently (interleaved) but ensures the final result is identical to at least one serial execution, providing better performance without sacrificing correctness.",
    "While Conflict Serializability focuses on the order of specific read/write conflicts, not all consistent schedules are conflict serializable. View Serializability is a broader concept that checks if the final outcome is equivalent to a serial schedule, ensuring consistency for a wider range of non-serial schedules.",
    "Implementing ACID properties can lead to performance overhead due to the extra processing required for consistency checks. Additionally, it can cause scalability issues in large distributed systems and increases the overall complexity of the DBMS architecture.",
    "While both have a growing phase to acquire locks, Strict-2PL does not have a typical shrinking phase where locks are released one by one. Instead, it holds all exclusive locks until the transaction either commits or aborts, specifically to prevent the \"Dirty Read\" problem.",
    "This scenario is handled by the Atomicity and Consistency properties. Because the transaction was not completed in its entirety, the \"All or nothing rule\" applies, and the database must be rolled back to its previous consistent state where the money is still in Account A.",
    "Under 2PL, the first agent's transaction must acquire an Exclusive Lock (X-Lock) on the seat data. The second agent's request for a lock will be blocked until the first agent's transaction commits and releases the lock, forcing the second agent to see the updated seat count of zero.",
    "This is the \"Inconsistent Analysis Problem\" (or Non-Repeatable Read). The manager's report (T1) got two different \"snapshots\" of the data at different times because Transaction 2 (the transfer) committed its changes while T1 was still running.",
    "In the analogy, the Growing Phase is like walking through the store and putting items (locks) into your cart. The \"Point of No Return\" or checkout represents the end of the growing phase; once the cashier starts scanning (Shrinking Phase), you can release items but cannot go back to grab more.",
    "The system is in a \"Deadlock.\" To resolve this, the system can use Deadlock Detection to identify the circular wait in a wait-for graph and then choose one transaction as a \"victim\" to be aborted, thereby releasing its locks and allowing the other to proceed.",
    "I don't know, or this information is not covered in this document. The provided text focuses on transaction processing and concurrency control, not normalization forms.",
    "I don't know, or this information is not covered in this document. The source material does not contain information on SQL syntax or aggregate query construction.",
    "I don't know, or this information is not covered in this document. This document deals with transactions and locking protocols rather than physical storage structures or indexing methods.",
    "I don't know, or this information is not covered in this document. Relational algebra operations and join types are not discussed in these slides.",
    "I don't know, or this information is not covered in this document. The source material focuses on concurrency and transaction states rather than database security or access control models.",
]

In [20]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


generated_answers = []
retrieved_contexts = []

for question in questions:
    contexts = retriever.invoke(question)

    answer = question_answer_chain.invoke({
        "input": question,
        "chat_history": [],
        "context": contexts,
    })

    generated_answers.append(answer)
    retrieved_contexts.append([doc.page_content for doc in contexts])

# =============================================================================
# RAGAS EVALUATION
# =============================================================================

judge_llm = LangchainLLMWrapper(llm)
judge_embeddings = LangchainEmbeddingsWrapper(embedding_model)

dataset = Dataset.from_dict({
    "question":     questions,
    "answer":       generated_answers,
    "contexts":     retrieved_contexts,
    "ground_truth": expected_answers,
})

results = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=judge_llm,
    embeddings=judge_embeddings,
)

print(results)
results.to_pandas().to_csv("ragas_results.csv", index=False)

/tmp/ipykernel_112563/3591897241.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_112563/3591897241.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_112563/3591897241.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import fait

{'faithfulness': 0.4444, 'answer_relevancy': 0.6413, 'context_precision': 0.6667, 'context_recall': 0.8667}


In [ ]:
chat_history=[]

In [ ]:
from langchain_core.messages import AIMessage,HumanMessage
response = rag_chain.invoke({
    'input':query,
    'chat_history':chat_history
})
chat_history.extend(
    [
        HumanMessage(content=query),
        AIMessage(content=response)
    ]
)

In [ ]:
response